In [ ]:
import scipy as sp
import numpy as np
import thewalrus as wr
import functools
import itertools
from tqdm.auto import tqdm
import bqplot.pyplot as plt
import bqplot as bq
import ipywidgets as widgets

In [ ]:
def T(n):
    """Defines conversion matrix for thewalrus
    T.T@sigma@T is our converted matrix
    T@sigma@T.T converts back
    """
    v1 = np.array([[1,0]])
    v2 = np.array([[0,1]])
    T1 = sp.linalg.block_diag(*([v1]*n))
    T2 = sp.linalg.block_diag(*([v2]*n))
    T = np.block([[T1],[T2]])
    return T
def σ(η,ns,nb):
    return np.array([
        [1 +2*η*ns + 2*nb,0,-2*np.sqrt(ns*(1+ns)*η),0],
        [0,1 +2*η*ns + 2*nb,0,2*np.sqrt(ns*(1+ns)*η)],
        [-2*np.sqrt(ns*(1+ns)*η),0,1 +2*ns,0],
        [0,2*np.sqrt(ns*(1+ns)*η),0,1 +2*ns]
    ])
def cov(η,ns,nb):
    return T(2)@σ(η,ns,nb)@T(2)
@functools.cache
def Ps_pn(η,ns,nb,maxval=10):
    means = np.zeros(4)
    covval = cov(η,ns,nb)
    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
    return Ps
def Ps_pnij(η,ns,nb,i,j,maxval=10):
    return Ps_pn(η,ns,nb,maxval)[i][j]
vec_Ps_pn = np.vectorize(Ps_pnij)
def to_diff_pn(ηs,*args):
    ns = args[0]
    nb = args[1]
    i = args[2]
    j = args[3]
    maxval=args[4]
    return vec_Ps_pn(ηs,ns,nb,i,j,maxval)
def FI_pn(ηs,nss,nbs,maxval=10):
    indices = np.arange(maxval)
    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
    Ps = vec_Ps_pn(ηgrid,nsgrid,nbgrid,igrid,jgrid,maxval)
    ds = np.zeros((len(ηs),len(nss),len(nbs),maxval,maxval))
    for i in range(maxval):
        for j in range(maxval):
            for k,ns in enumerate(nss):
                for l,nb in enumerate(nbs):
                    derivres = sp.differentiate.derivative(to_diff_pn,ηvals,args = [ns,nb,i,j,maxval],initial_step=1e-5)
                    ds[:,k,l,i,j] = derivres.df
    presum = ds**2/Ps
    presum[~np.isfinite(presum)] = 0
    FI = np.sum(presum,axis=(3,4))
    return FI


def σsu(η,ns,nb):
    bkrd = 1+2*ns*((1+ns)*(1+np.sqrt(η))**2+nb)
    corr = 2*np.sqrt(ns*(1+ns))*(1+np.sqrt(η)+nb+ns*(1+np.sqrt(η))**2)
    return np.array([
        [bkrd + 2*nb,0,-corr,0],
        [0,bkrd+ 2*nb,0,corr],
        [-corr,0,bkrd+2*ns*(1-η),0],
        [0,corr,0,bkrd+2*ns*(1-η)]
    ])
def covsu(η,ns,nb):
    return T(2)@σsu(η,ns,nb)@T(2)
@functools.cache
def Ps_su(η,ns,nb,maxval=10):
    means = np.zeros(4)
    covval = covsu(η,ns,nb)
    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
    return Ps
def Ps_suij(η,ns,nb,i,j,maxval=10):
    return Ps_su(η,ns,nb,maxval)[i][j]

vec_Ps_su = np.vectorize(Ps_suij)


def to_diff_su(ηs,*args):
    ns = args[0]
    nb = args[1]
    i = args[2]
    j = args[3]
    maxval = args[4]
    return vec_Ps_su(ηs,ns,nb,i,j,maxval)
def FI_su(ηs,nss,nbs,maxval=10):
    indices = np.arange(maxval)
    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
    Ps = vec_Ps_su(ηgrid,nsgrid,nbgrid,igrid,jgrid,maxval)
    ds = np.zeros((len(ηs),len(nss),len(nbs),maxval,maxval))
    for i in range(maxval):
        for j in range(maxval):
            for k,ns in enumerate(nss):
                for l,nb in enumerate(nbs):
                    derivres = sp.differentiate.derivative(to_diff_su,ηvals,args = [ns,nb,i,j,maxval],initial_step=1e-5)
                    ds[:,k,l,i,j] = derivres.df
    presum = ds**2/Ps
    presum[~np.isfinite(presum)] = 0
    FI = np.sum(presum,axis=(3,4))
    return FI



def Ps_pn_jac(η,*args):
    ns,nb = args[0,1]
    maxval = args[2]
    means = np.zeros(4)
    covval = cov(η,ns,nb)
    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
    return Ps
def vec_Ps_pn_jac_gen(*args):
    def vec_Ps_pn_jac(x):
        return np.apply_along_axis(Ps_pn_jac, axis=0, arr=x,args = args)
    return vec_Ps_pn_jac
def FI_pn_jac(ηs,nss,nbs,maxval=10):
    indices = np.arange(maxval)
    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
    Ps = vec_Ps_pn(ηgrid,nsgrid,nbgrid,igrid,jgrid,maxval)
    ds = np.zeros((len(ηs),len(nss),len(nbs),maxval,maxval))
    for i,ns in enumerate(nss):
        for j,nb in enumerate(nbs):
            vec_Ps_pn_jac = vec_Ps_pn_jac_gen(ns,nb,maxval)
            derivres = sp.differentiate.jacobian(vec_Ps_pn_jac,ηs,initial_step=1e-5)
            ds[:,i,j,:,:] = derivres,df
    presum = ds**2/Ps
    presum[~np.isfinite(presum)] = 0
    FI = np.sum(presum,axis=(3,4))
    return FI    
def Ps_su_jac(η,args):
    ns,nb = args[0:2]
    maxval = args[2]
    means = np.zeros(4)
    covval = covsu(η,ns,nb)
    Ps = wr.quantum.probabilities(means,covval,maxval,atol=1e-10,rtol=1e-7)
    return Ps
def vec_Ps_su_jac_gen(*args):
    def vec_Ps_su_jac(x):
        return np.apply_along_axis(Ps_su_jac, axis=0, arr=x,args = args)
    return vec_Ps_su_jac
def FI_su_jac(ηs,nss,nbs,maxval=10):
    indices = np.arange(maxval)
    ηgrid,nsgrid,nbgrid,igrid,jgrid = np.meshgrid(ηs,nss,nbs,indices,indices,indexing='ij')
    ds = np.zeros((len(ηs),len(nss),len(nbs),maxval,maxval))
    for (i,ns),(j,nb) in tqdm(itertools.product(enumerate(nss),enumerate(nbs)),total=len(nss)*len(nbs),smoothing=.01):
        vec_Ps_su_jac = vec_Ps_su_jac_gen(ns,nb,maxval)
        derivres = sp.differentiate.jacobian(vec_Ps_su_jac,ηs,initial_step=1e-5)
        ds[:,i,j,:,:] = derivres,df
    Ps = vec_Ps_su(ηgrid,nsgrid,nbgrid,igrid,jgrid,maxval)
    presum = ds**2/Ps
    presum[~np.isfinite(presum)] = 0
    FI = np.sum(presum,axis=(3,4))
    return FI

In [ ]:
ηvals = np.linspace(1e-3,1-1e-3,80,dtype=np.double)
nsvals = np.logspace(-4,.5,10,dtype=np.double)
nbvals = np.insert(np.logspace(-3,-.5,9,dtype=np.double),0,0)
maxval = 10

In [ ]:
#vec_Ps_su_jac_gen(.1,.1,10)(np.array([.1]))
Ps_su_jac(.1,[.1,.1,10])

In [ ]:
#FIspn = FI_pn(ηvals,nsvals,nbvals,maxval)
FIssu = FI_su_jac(ηvals,nsvals,nbvals,maxval)

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
ax.plot(ηvals,FIspn[:,4,8],label="Biphoton")
ax.plot(ηvals,FIssu[:,4,8],label="SU(1,1)")
ax.set_xlabel('eta')
ax.set_ylabel('FI')
plt.legend(loc='best')
plt.show()

In [ ]:
fig = plt.figure()
ax = fig.add_subplot()
ax.plot(ηvals,FIssu[:,4,8]-FIspn[:,4,8],label="Difference")
ax.set_xlabel('eta')
ax.set_ylabel('FI')
plt.legend(loc='best')
plt.show()

In [ ]:
nsslider = widgets.IntSlider(
    value=7,
    min=0,
    max=maxval-1,
    step=1,
    description='Ns index',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
nbslider = widgets.IntSlider(
    value=7,
    min=0,
    max=maxval-1,
    step=1,
    description='Nb index:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
def display_graph(nsi,nbi):
    fig = plt.figure()
    ax = fig.add_subplot()
    ax.plot(ηvals,FIssu[:,nsi,nbi]-FIspn[:,nsi,nbi],label="Difference")
    ax.set_xlabel('eta')
    ax.set_ylabel('FI')
    plt.legend(loc='best')
    return fig


In [ ]:
widgets.interact(display_graph,nsi = nsslider,nbi = nbslider,continuous=False)

In [ ]:
class Selectable_plot(widgets.VBox):


    def __init__(self, xdata,ydata, **kwargs):
        self.data = data

        self.x_scale = bq.LinearScale()
        self.y_scale = bq.LinearScale()

        self.x_ax = bq.Axis(scale=self.x_scale)
        self.y_ax = bq.Axis(scale=self.y_scale, orientation="vertical")
        show_axes = kwargs.pop("show_axes", True)
        self.axes = [self.x_ax, self.y_ax] if show_axes is True else []

        self.height = kwargs.pop("height", "800px")
        self.layout = kwargs.pop(
            "layout", Layout(width="100%", height=self.height, flex="1")
        )
        self.fig_margin = kwargs.pop(
            "fig_margin", {"top": 60, "bottom": 60, "left": 150, "right": 0}
        )
        kwargs.setdefault("padding_y", 0.0)

        self.create_interaction(**kwargs)
        self.line = bq.Lines(
            x=data[0,0],
            y=ydata,
            scales={"x": self.x_scale, "y": self.y_scale})

        self.figure = bq.Figure(
            marks=[self.line],
            axes=self.axes,
            fig_margin=self.fig_margin,
            layout=self.layout,
            min_aspect_ratio=0.0,
            **kwargs
        )

        super(widgets.VBox, self).__init__(
            children=[self.nb_slider,self.ns_slider, self.figure],
            layout=widgets.Layout(align_items="center", width="100%", height="100%"),
            **kwargs
        )

    def create_interaction(self, **kwargs):
        self.nb_slider = widgets.IntRangeSlider(
            description="nb index",
            value=(0, self.nb_range),
            layout=widgets.Layout(width="100%"),
        )
        self.ns_slider = widgets.IntRangeSlider(
            description="ns inder",
            value=(0, self.ns_range),
            layout=widgets.Layout(width="100%"),
        )
        self.nb_slider.observe(self.nbslid_changed, "value")
        self.ns_slider.observe(self.nsslid_changed, "value")
        self.observe(self.changed, ["nbi", "nsi"])

    def nbslid_changed(self, new):
        self.nbi = self.nb_slider.value
    def nsslid_changed(self, new):
        self.nsi = self.ns_slider.value

    def changed(self, new):
        self.nbslider.value = self.nbi
        self.nsslider.value = self.nsi
        self.line.x = xdata[self.nsi,self.nbi]



In [ ]:
selectable_plot = Selectable_plot(
    FIssu-FIspu,ηvals, title="Fisher Difference", height="1400px", colors=["Red", "White", "Green"]
)
selectable_plot